# Quantum Error Correction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/03_quantum_error_correction.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/03_quantum_error_correction.ipynb)


This notebook explores:
1. **Repetition code** — syndrome extraction and correction
2. **Surface code threshold** — logical error rate vs. code distance
3. **Quantum LDPC codes** — hypergraph product code construction


In [ ]:
# Install quantx-engine (needed in Colab / Binder)
# Skipped automatically if already installed in the local environment.
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

## 1. Distance-3 Repetition Code

The distance-3 repetition code corrects all single-qubit bit-flip errors.

The `RepetitionCode` API:
- `encode(logical: int) -> list[int]`
- `syndrome(codeword) -> list[int]`
- `correct(codeword) -> list[int]` — takes noisy codeword, returns corrected codeword
- `decode(codeword) -> int`

In [ ]:
from quantx_engine.qec import RepetitionCode

code = RepetitionCode(n=3)

# Encode logical 0 (integer, not list)
encoded0 = code.encode(0)
print(f'Logical 0 encoded: {encoded0}')

# Test all single-qubit bit-flip errors on logical 0
for err_qubit in range(3):
    noisy = list(encoded0)
    noisy[err_qubit] ^= 1
    corrected = code.correct(noisy)
    decoded   = code.decode(corrected)
    status = 'OK' if decoded == 0 else 'FAIL'
    print(f'  Error on qubit {err_qubit}: noisy={noisy} -> corrected={corrected} -> decoded={decoded}  [{status}]')
    assert decoded == 0, f'Correction failed for qubit {err_qubit}'

In [ ]:
# Test logical 1 encoding and correction
encoded1 = code.encode(1)
print(f'Logical 1 encoded: {encoded1}')

for err_qubit in range(3):
    noisy = list(encoded1)
    noisy[err_qubit] ^= 1
    corrected = code.correct(noisy)
    decoded   = code.decode(corrected)
    assert decoded == 1, f'Logical 1 correction failed for qubit {err_qubit}'
print('All weight-1 errors corrected for both logical 0 and logical 1.')

## 2. Surface Code Threshold

Below the threshold, increasing the code distance reduces the logical error rate.

In [ ]:
from quantx_engine.fault_tolerant import logical_error_rate_vs_distance

p_low  = 0.001   # well below threshold
p_high = 0.15    # above threshold

rates_low  = logical_error_rate_vs_distance([3, 5, 7], p_low,  n_shots=300, rng_seed=42)
rates_high = logical_error_rate_vs_distance([3, 5, 7], p_high, n_shots=300, rng_seed=42)

print(f'p_physical = {p_low}  (below threshold)')
for d, r in rates_low.items():
    print(f'  d={d}: logical rate = {r:.5f}')

print(f'\np_physical = {p_high}  (above threshold)')
for d, r in rates_high.items():
    print(f'  d={d}: logical rate = {r:.5f}')

# Below threshold: d=5 should be at least as good as d=3
assert rates_low[5] <= rates_low[3] + 0.01, \
    'Below threshold: d=5 should not be worse than d=3'
print('\nThreshold behaviour confirmed.')

## 3. Quantum LDPC Codes (M119)

The `qldpc` module implements quantum LDPC codes via the hypergraph product construction.  
We build a hypergraph-product code from two classical repetition codes.

In [ ]:
from quantx_engine.qldpc import ClassicalCode, HypergraphProductCode

# Build a hypergraph product [[n,k,d]] code from two repetition codes
# ClassicalCode.repetition(n) creates the n-bit repetition classical code
c = ClassicalCode.repetition(6)
print(f'Classical repetition code: [n={c.n}, m={c.m}, k={c.k}]')

qcode = HypergraphProductCode(c, c)
print(f'Hypergraph product code:   n_qubits={qcode.n_qubits}, k={qcode.k}')
print(f'Hx shape: {qcode.Hx.shape},  Hz shape: {qcode.Hz.shape}')

# Verify the fundamental CSS property: Hx @ Hz.T = 0 (mod 2)
commutes = qcode.commutation_check()
print(f'CSS commutation check (Hx Hz^T = 0 mod 2): {commutes}')
assert commutes, 'CSS commutation violated!'
assert qcode.k > 0, 'Code must encode at least one logical qubit'
print('LDPC code assertions passed.')

## Summary

| Module | Demonstrated |
|--------|--------------|
| `qec.RepetitionCode` | Encode, correct, decode — all weight-1 errors on d=3 code |
| `fault_tolerant.logical_error_rate_vs_distance` | Threshold behaviour: d=5 ≤ d=3 below threshold |
| `qldpc.HypergraphProductCode` | CSS QLDPC code from two repetition codes |

Next: **[04 — Quantum Networking](04_quantum_networking.ipynb)**